# SASRec Delta-Start Time-Aware BPI2012 Colab Train Combo (`refine_ml50_do035` baseline)

Colab notebook for comparing two `delta_start_seconds` time-aware variants on top of `refine_ml50_do035`:
- `delta_start_seconds + 9-bucket`
- `delta_start_seconds + continuous`

Both are evaluated under `NDCG@10` and `NDCG@5` model-selection criteria.


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
BUCKET_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10'
BUCKET_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5'
CONT_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10'
CONT_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('BUCKET_NDCG10_OUTPUT_DIR:', BUCKET_NDCG10_OUTPUT_DIR)
print('BUCKET_NDCG5_OUTPUT_DIR:', BUCKET_NDCG5_OUTPUT_DIR)
print('CONT_NDCG10_OUTPUT_DIR:', CONT_NDCG10_OUTPUT_DIR)
print('CONT_NDCG5_OUTPUT_DIR:', CONT_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
BUCKET_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10
BUCKET_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5
CONT_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10
CONT_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$BUCKET_NDCG10_OUTPUT_DIR"
!mkdir -p "$BUCKET_NDCG5_OUTPUT_DIR"
!mkdir -p "$CONT_NDCG10_OUTPUT_DIR"
!mkdir -p "$CONT_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Time-aware variants:
- `delta_start_seconds + 9-bucket`
- `delta_start_seconds + continuous`


## Check baseline runs


In [10]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR)),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR)),
]:
    print('=' * 80)
    print(label)
    for run_name in baseline_runs:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Baseline NDCG@5
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned new runs


In [11]:
planned_bucket_ndcg10 = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
planned_bucket_ndcg5 = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
planned_cont_ndcg10 = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]
planned_cont_ndcg5 = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

for label, output_dir, run_names in [
    ('Bucket NDCG@10', Path(BUCKET_NDCG10_OUTPUT_DIR), planned_bucket_ndcg10),
    ('Bucket NDCG@5', Path(BUCKET_NDCG5_OUTPUT_DIR), planned_bucket_ndcg5),
    ('Continuous NDCG@10', Path(CONT_NDCG10_OUTPUT_DIR), planned_cont_ndcg10),
    ('Continuous NDCG@5', Path(CONT_NDCG5_OUTPUT_DIR), planned_cont_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Bucket NDCG@10
timeaware_dstart_refine_ml50_do035_b9_s42 OK
timeaware_dstart_refine_ml50_do035_b9_s2024 OK
timeaware_dstart_refine_ml50_do035_b9_s7 OK
Bucket NDCG@5
timeaware_dstart_refine_ml50_do035_b9_s42 OK
timeaware_dstart_refine_ml50_do035_b9_s2024 OK
timeaware_dstart_refine_ml50_do035_b9_s7 OK
Continuous NDCG@10
timeaware_dstart_conti_refine_ml50_do035_s42 OK
timeaware_dstart_conti_refine_ml50_do035_s2024 OK
timeaware_dstart_conti_refine_ml50_do035_s7 OK
Continuous NDCG@5
timeaware_dstart_conti_refine_ml50_do035_s42 OK
timeaware_dstart_conti_refine_ml50_do035_s2024 OK
timeaware_dstart_conti_refine_ml50_do035_s7 OK


## Train `delta_start + 9-bucket` for `NDCG@10`


### timeaware_dstart_refine_ml50_do035_b9_s42


In [12]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10/timeaware_dstart_refine_ml50_do035_b9_s42
epoch=1, loss=0.5836
epoch=2, loss=0.2676
epoch=3, loss=0.2017
epoch=4, loss=0.1652
epoch=5, loss=0.1407
valid [full], NDCG@5: 0.6477, HR@5: 0.7565, NDCG@10: 0.6925, HR@10: 0.8994, MRR: 0.6391
valid [sampled], NDCG@5: 0.5589, HR@5: 0.5685, NDCG@10: 0.5716, HR@10: 0.6080, MRR: 0.5732
test [full], NDCG@5: 0.5774, HR@5: 0.7196, NDCG@10: 0.6227, HR@10: 0.8644, MRR: 0.5576
test [sampled], NDCG@5: 0.1890, HR@5: 0.1942, NDCG@10: 0.2341, HR@10: 0.3400, MRR: 0.2338
saved eval checkpoint: /content/dri

### timeaware_dstart_refine_ml50_do035_b9_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10/timeaware_dstart_refine_ml50_do035_b9_s2024
epoch=1, loss=0.6049
epoch=2, loss=0.2704
epoch=3, loss=0.1983
epoch=4, loss=0.1600
epoch=5, loss=0.1394
valid [full], NDCG@5: 0.7006, HR@5: 0.8868, NDCG@10: 0.7361, HR@10: 0.9909, MRR: 0.6582
valid [sampled], NDCG@5: 0.5591, HR@5: 0.5649, NDCG@10: 0.5684, HR@10: 0.5942, MRR: 0.5768
test [full], NDCG@5: 0.7452, HR@5: 0.8726, NDCG@10: 0.7861, HR@10: 1.0000, MRR: 0.7202
test [sampled], NDCG@5: 0.2070, HR@5: 0.2670, NDCG@10: 0.2638, HR@10: 0.4422, MRR: 0.2392
saved eval checkpoint: /content/d

### timeaware_dstart_refine_ml50_do035_b9_s7


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10/timeaware_dstart_refine_ml50_do035_b9_s7
epoch=1, loss=0.5819
epoch=2, loss=0.2724
epoch=3, loss=0.2027
epoch=4, loss=0.1694
epoch=5, loss=0.1482
valid [full], NDCG@5: 0.6387, HR@5: 0.7412, NDCG@10: 0.6910, HR@10: 0.9039, MRR: 0.6356
valid [sampled], NDCG@5: 0.5609, HR@5: 0.5617, NDCG@10: 0.5681, HR@10: 0.5849, MRR: 0.5771
test [full], NDCG@5: 0.6286, HR@5: 0.8319, NDCG@10: 0.6540, HR@10: 0.9095, MRR: 0.5786
test [sampled], NDCG@5: 0.2090, HR@5: 0.2122, NDCG@10: 0.2408, HR@10: 0.3155, MRR: 0.2545
saved eval checkpoint: /content/driv

## Train `delta_start + continuous` for `NDCG@10`


### timeaware_dstart_conti_refine_ml50_do035_s42


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10/timeaware_dstart_conti_refine_ml50_do035_s42
epoch=1, loss=0.6435
epoch=2, loss=0.3108
epoch=3, loss=0.2256
epoch=4, loss=0.1818
epoch=5, loss=0.1542
valid [full], NDCG@5: 0.5792, HR@5: 0.6165, NDCG@10: 0.6660, HR@10: 0.8930, MRR: 0.6096
valid [sampled], NDCG@5: 0.5436, HR@5: 0.5451, NDCG@10: 0.5508, HR@10: 0.5682, MRR: 0.5565
test [full], NDCG@5: 0.5982, HR@5: 0.7511, NDCG@10: 0.6383, HR@10: 0.8839, MRR: 0.5684
test [sampled], NDCG@5: 0.1847, HR@5: 0.1933, NDCG@10: 0.2183, HR@10: 0.3020, MRR: 0.2273
saved eval checkpoint: /

### timeaware_dstart_conti_refine_ml50_do035_s2024


In [16]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10/timeaware_dstart_conti_refine_ml50_do035_s2024
epoch=1, loss=0.6747
epoch=2, loss=0.3270
epoch=3, loss=0.2288
epoch=4, loss=0.1837
epoch=5, loss=0.1540
valid [full], NDCG@5: 0.6732, HR@5: 0.7996, NDCG@10: 0.7262, HR@10: 0.9722, MRR: 0.6549
valid [sampled], NDCG@5: 0.5662, HR@5: 0.5775, NDCG@10: 0.5785, HR@10: 0.6155, MRR: 0.5806
test [full], NDCG@5: 0.7545, HR@5: 0.8810, NDCG@10: 0.7927, HR@10: 1.0000, MRR: 0.7281
test [sampled], NDCG@5: 0.1394, HR@5: 0.2107, NDCG@10: 0.2161, HR@10: 0.4479, MRR: 0.1762
saved eval checkpoint:

### timeaware_dstart_conti_refine_ml50_do035_s7


In [17]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10/timeaware_dstart_conti_refine_ml50_do035_s7
epoch=1, loss=0.6368
epoch=2, loss=0.3261
epoch=3, loss=0.2338
epoch=4, loss=0.1884
epoch=5, loss=0.1637
valid [full], NDCG@5: 0.6210, HR@5: 0.7248, NDCG@10: 0.6889, HR@10: 0.9404, MRR: 0.6202
valid [sampled], NDCG@5: 0.5455, HR@5: 0.5473, NDCG@10: 0.5513, HR@10: 0.5656, MRR: 0.5609
test [full], NDCG@5: 0.5974, HR@5: 0.6974, NDCG@10: 0.6735, HR@10: 0.9348, MRR: 0.5999
test [sampled], NDCG@5: 0.1973, HR@5: 0.2229, NDCG@10: 0.2381, HR@10: 0.3519, MRR: 0.2324
saved eval checkpoint: /c

## Train `delta_start + 9-bucket` for `NDCG@5`


### timeaware_dstart_refine_ml50_do035_b9_s42


In [18]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5/timeaware_dstart_refine_ml50_do035_b9_s42
epoch=1, loss=0.5836
epoch=2, loss=0.2676
epoch=3, loss=0.2017
epoch=4, loss=0.1652
epoch=5, loss=0.1407
valid [full], NDCG@5: 0.6477, HR@5: 0.7565, NDCG@10: 0.6925, HR@10: 0.8994, MRR: 0.6391
valid [sampled], NDCG@5: 0.5589, HR@5: 0.5685, NDCG@10: 0.5716, HR@10: 0.6080, MRR: 0.5732
test [full], NDCG@5: 0.5774, HR@5: 0.7196, NDCG@10: 0.6227, HR@10: 0.8644, MRR: 0.5576
test [sampled], NDCG@5: 0.1890, HR@5: 0.1942, NDCG@10: 0.2341, HR@10: 0.3400, MRR: 0.2338
saved eval checkpoint: /content/drive

### timeaware_dstart_refine_ml50_do035_b9_s2024


In [19]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5/timeaware_dstart_refine_ml50_do035_b9_s2024
epoch=1, loss=0.6049
epoch=2, loss=0.2704
epoch=3, loss=0.1983
epoch=4, loss=0.1600
epoch=5, loss=0.1394
valid [full], NDCG@5: 0.7006, HR@5: 0.8868, NDCG@10: 0.7361, HR@10: 0.9909, MRR: 0.6582
valid [sampled], NDCG@5: 0.5591, HR@5: 0.5649, NDCG@10: 0.5684, HR@10: 0.5942, MRR: 0.5768
test [full], NDCG@5: 0.7452, HR@5: 0.8726, NDCG@10: 0.7861, HR@10: 1.0000, MRR: 0.7202
test [sampled], NDCG@5: 0.2070, HR@5: 0.2670, NDCG@10: 0.2638, HR@10: 0.4422, MRR: 0.2392
saved eval checkpoint: /content/dri

### timeaware_dstart_refine_ml50_do035_b9_s7


In [20]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5/timeaware_dstart_refine_ml50_do035_b9_s7
epoch=1, loss=0.5819
epoch=2, loss=0.2724
epoch=3, loss=0.2027
epoch=4, loss=0.1694
epoch=5, loss=0.1482
valid [full], NDCG@5: 0.6387, HR@5: 0.7412, NDCG@10: 0.6910, HR@10: 0.9039, MRR: 0.6356
valid [sampled], NDCG@5: 0.5609, HR@5: 0.5617, NDCG@10: 0.5681, HR@10: 0.5849, MRR: 0.5771
test [full], NDCG@5: 0.6286, HR@5: 0.8319, NDCG@10: 0.6540, HR@10: 0.9095, MRR: 0.5786
test [sampled], NDCG@5: 0.2090, HR@5: 0.2122, NDCG@10: 0.2408, HR@10: 0.3155, MRR: 0.2545
saved eval checkpoint: /content/drive/

## Train `delta_start + continuous` for `NDCG@5`


### timeaware_dstart_conti_refine_ml50_do035_s42


In [21]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5/timeaware_dstart_conti_refine_ml50_do035_s42
epoch=1, loss=0.6435
epoch=2, loss=0.3108
epoch=3, loss=0.2256
epoch=4, loss=0.1818
epoch=5, loss=0.1542
valid [full], NDCG@5: 0.5792, HR@5: 0.6165, NDCG@10: 0.6660, HR@10: 0.8930, MRR: 0.6096
valid [sampled], NDCG@5: 0.5436, HR@5: 0.5451, NDCG@10: 0.5508, HR@10: 0.5682, MRR: 0.5565
test [full], NDCG@5: 0.5982, HR@5: 0.7511, NDCG@10: 0.6383, HR@10: 0.8839, MRR: 0.5684
test [sampled], NDCG@5: 0.1847, HR@5: 0.1933, NDCG@10: 0.2183, HR@10: 0.3020, MRR: 0.2273
saved eval checkpoint: /co

### timeaware_dstart_conti_refine_ml50_do035_s2024


In [22]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5/timeaware_dstart_conti_refine_ml50_do035_s2024
epoch=1, loss=0.6747
epoch=2, loss=0.3270
epoch=3, loss=0.2288
epoch=4, loss=0.1837
epoch=5, loss=0.1540
valid [full], NDCG@5: 0.6732, HR@5: 0.7996, NDCG@10: 0.7262, HR@10: 0.9722, MRR: 0.6549
valid [sampled], NDCG@5: 0.5662, HR@5: 0.5775, NDCG@10: 0.5785, HR@10: 0.6155, MRR: 0.5806
test [full], NDCG@5: 0.7545, HR@5: 0.8810, NDCG@10: 0.7927, HR@10: 1.0000, MRR: 0.7281
test [sampled], NDCG@5: 0.1394, HR@5: 0.2107, NDCG@10: 0.2161, HR@10: 0.4479, MRR: 0.1762
saved eval checkpoint: /

### timeaware_dstart_conti_refine_ml50_do035_s7


In [23]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5/timeaware_dstart_conti_refine_ml50_do035_s7
epoch=1, loss=0.6368
epoch=2, loss=0.3261
epoch=3, loss=0.2338
epoch=4, loss=0.1884
epoch=5, loss=0.1637
valid [full], NDCG@5: 0.6210, HR@5: 0.7248, NDCG@10: 0.6889, HR@10: 0.9404, MRR: 0.6202
valid [sampled], NDCG@5: 0.5455, HR@5: 0.5473, NDCG@10: 0.5513, HR@10: 0.5656, MRR: 0.5609
test [full], NDCG@5: 0.5974, HR@5: 0.6974, NDCG@10: 0.6735, HR@10: 0.9348, MRR: 0.5999
test [sampled], NDCG@5: 0.1973, HR@5: 0.2229, NDCG@10: 0.2381, HR@10: 0.3519, MRR: 0.2324
saved eval checkpoint: /con

## Rebuild result tables


In [24]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_feature_dim': config.get('time_feature_dim'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [30]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [31]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
bucket_runs = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
continuous_runs = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
bucket_df = rebuild_df(BUCKET_NDCG10_OUTPUT_DIR)
continuous_df = rebuild_df(CONT_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

bucket_subset = bucket_df[bucket_df['run_name'].isin(bucket_runs)].copy()
bucket_subset['time_variant'] = 'delta_start_b9'

continuous_subset = continuous_df[continuous_df['run_name'].isin(continuous_runs)].copy()
continuous_subset['time_variant'] = 'delta_start_continuous'

df_ndcg10 = pd.concat([baseline_subset, bucket_subset, continuous_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'time_encoding', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,time_encoding,time_delta_column,time_bucket_boundaries_parsed,time_feature_dim,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,refine_ml50_do035_s7,7,baseline,None,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
1,refine_ml50_do035_s42,42,baseline,None,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
2,refine_ml50_do035_s2024,2024,baseline,None,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
3,timeaware_dstart_refine_ml50_do035_b9_s7,7,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.745142,0.957660,0.717650,0.865653,0.682456,0.913922,1.000000,0.913685,0.999321,0.884692,0.590203,0.671790,0.558999,0.572633,0.580722,0.430771,0.644776,0.359573,0.424016,0.390789
4,timeaware_dstart_refine_ml50_do035_b9_s42,42,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.728625,0.964446,0.681780,0.820464,0.659430,0.663047,1.000000,0.649231,0.958485,0.549901,0.588615,0.633023,0.572991,0.584380,0.589039,0.181912,0.318336,0.125099,0.135517,0.181054
5,timeaware_dstart_refine_ml50_do035_b9_s2024,2024,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.752983,0.972787,0.721568,0.873097,0.686346,0.827690,0.999865,0.811716,0.948416,0.770331,0.592942,0.678122,0.560615,0.576106,0.582732,0.366431,0.517942,0.306744,0.327014,0.352601
6,timeaware_dstart_conti_refine_ml50_do035_s7,7,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.742792,0.965143,0.697805,0.822732,0.676947,0.667157,0.999865,0.666316,0.997428,0.554053,0.590123,0.664322,0.561008,0.571950,0.582827,0.138163,0.283937,0.076503,0.084911,0.138319
7,timeaware_dstart_conti_refine_ml50_do035_s42,42,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.737949,0.971231,0.708916,0.877731,0.668492,0.703322,1.000000,0.692492,0.968087,0.604422,0.596241,0.646264,0.578629,0.591136,0.595470,0.234225,0.398668,0.174982,0.212315,0.217385
8,timeaware_dstart_conti_refine_ml50_do035_s2024,2024,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.732435,0.971097,0.700672,0.871556,0.661297,0.785443,1.000000,0.761312,0.920419,0.716429,0.566956,0.601116,0.554750,0.563146,0.573079,0.220623,0.473166,0.139440,0.221548,0.172588


In [32]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                          mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
baseline                              0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175
delta_start_b9                        0.742250  0.012434              0.964964  0.007577               0.706999  0.021928             0.853072  0.028483            0.676078  0.014548               0.801553  0.127464             0.999955  0.000078              0.791544  0.133376            0.968741  0.026958           0.734975  0.170173                   0.590587  0.002189                 0.660978  0.024416                  0.564202  0.007655                0.577707  0.006035               0.584164  0.004340                  0.326371  0.129175                0.493685  0.164566                 0.263805  0.122993               0.295516  0.146806              0.308148  0.111710
delta_start_continuous                0.737725  0.005182              0.969157  0.003477               0.702464  0.005768             0.857340  0.030130            0.668912  0.007834               0.718641  0.060612             0.999955  0.000078              0.706706  0.049067            0.961978  0.038866           0.624968  0.083115                   0.584440  0.015447                 0.637234  0.032556                  0.564796  0.012382                0.575411  0.014312               0.583792  0.011227                  0.197670  0.051982                0.385257  0.095325                 0.130308 

Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled and MRR move in the same direction
- this notebook compares `delta_start_seconds + 9-bucket` vs `delta_start_seconds + continuous` under the same baseline


## NDCG@5 comparison summary


In [33]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
bucket_runs = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
continuous_runs = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
bucket_df = rebuild_df(BUCKET_NDCG5_OUTPUT_DIR)
continuous_df = rebuild_df(CONT_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

bucket_subset = bucket_df[bucket_df['run_name'].isin(bucket_runs)].copy()
bucket_subset['time_variant'] = 'delta_start_b9'

continuous_subset = continuous_df[continuous_df['run_name'].isin(continuous_runs)].copy()
continuous_subset['time_variant'] = 'delta_start_continuous'

df_ndcg5 = pd.concat([baseline_subset, bucket_subset, continuous_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'time_encoding', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,time_encoding,time_delta_column,time_bucket_boundaries_parsed,time_feature_dim,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,refine_ml50_do035_s7,7,baseline,None,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
1,refine_ml50_do035_s42,42,baseline,None,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
2,refine_ml50_do035_s2024,2024,baseline,None,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
3,timeaware_dstart_refine_ml50_do035_b9_s7,7,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.744562,0.948239,0.718698,0.863539,0.684462,0.911379,1.000000,0.908915,0.992815,0.881427,0.593869,0.673816,0.563615,0.578662,0.584333,0.412148,0.631351,0.341570,0.413494,0.369849
4,timeaware_dstart_refine_ml50_do035_b9_s42,42,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.721305,0.936017,0.687943,0.830826,0.660536,0.713420,0.999865,0.702081,0.964030,0.617557,0.577894,0.626563,0.559777,0.569739,0.576792,0.214884,0.333649,0.166365,0.177576,0.220794
5,timeaware_dstart_refine_ml50_do035_b9_s2024,2024,delta_start_b9,bucket,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",1,0.752983,0.972787,0.721568,0.873097,0.686346,0.827690,0.999865,0.811716,0.948416,0.770331,0.592942,0.678122,0.560615,0.576106,0.582732,0.366431,0.517942,0.306744,0.327014,0.352601
6,timeaware_dstart_conti_refine_ml50_do035_s7,7,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.740357,0.954274,0.699253,0.822795,0.678027,0.830831,0.999864,0.829786,0.996744,0.772888,0.591938,0.667479,0.563096,0.576262,0.583469,0.227032,0.487845,0.138473,0.211870,0.182775
7,timeaware_dstart_conti_refine_ml50_do035_s42,42,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.737949,0.971231,0.708916,0.877731,0.668492,0.703322,1.000000,0.692492,0.968087,0.604422,0.596241,0.646264,0.578629,0.591136,0.595470,0.234225,0.398668,0.174982,0.212315,0.217385
8,timeaware_dstart_conti_refine_ml50_do035_s2024,2024,delta_start_continuous,continuous,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0]",2,0.732435,0.971097,0.700672,0.871556,0.661297,0.785443,1.000000,0.761312,0.920419,0.716429,0.566956,0.601116,0.554750,0.563146,0.573079,0.220623,0.473166,0.139440,0.221548,0.172588


In [34]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                          mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
baseline                              0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175
delta_start_b9                        0.739616  0.016408              0.952348  0.018726               0.709403  0.018640             0.855820  0.022168            0.677115  0.014388               0.817496  0.099373             0.999910  0.000078              0.807570  0.103479            0.968420  0.022523           0.756439  0.132482                   0.588235  0.008967                 0.659500  0.028605                  0.561336  0.002018                0.574836  0.004595               0.581286  0.003973                  0.331154  0.103255                0.494314  0.150251                 0.271559  0.092751               0.306028  0.119351              0.314415  0.081535
delta_start_continuous                0.736914  0.004061              0.965534  0.009751               0.702947  0.005218             0.857361  0.030093            0.669272  0.008392               0.773199  0.064631             0.999955  0.000078              0.761196  0.068647            0.961750  0.038555           0.697913  0.085746                   0.585045  0.015812                 0.638286  0.033893                  0.565492  0.012119                0.576848  0.014004               0.584006  0.011205                  0.227293  0.006805                0.453226  0.047815                 0.150965 

Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled and MRR move in the same direction
- this notebook compares `delta_start_seconds + 9-bucket` vs `delta_start_seconds + continuous` under the same baseline
